In [1]:
import os, json, glob, pandas as pd, numpy as np
base='/workspace'
for f in ['campaign_metrics.json','partial_sessions_metrics.json','params.json']:
    p=os.path.join(base,f)
    with open(p) as fh: obj=json.load(fh)
    print(f, type(obj), (list(obj)[:10] if isinstance(obj,dict) else len(obj)))
    if isinstance(obj,dict):
        for k,v in list(obj.items())[:2]: print(' ',k,type(v), (len(v) if hasattr(v,'__len__') else None))
print('\nSummary CSV:')
df=pd.read_csv(os.path.join(base,'t3-prism-bo-batch-drop-results.csv'))
print(df.shape); print(df.to_string(index=False))

campaign_metrics.json <class 'dict'> ['config', 'warmup_discarded_per_specimen', 'v_freefall_ms', 'dv_references', 'specimens', 'campaign']
  config <class 'str'> 78
  warmup_discarded_per_specimen <class 'int'> None
partial_sessions_metrics.json <class 'dict'> ['config', 'warmup_discarded_per_specimen', 'v_freefall_ms', 'dv_references', 'specimens', 'campaign']
  config <class 'str'> 78
  warmup_discarded_per_specimen <class 'int'> None
params.json <class 'dict'> ['bpx68c', '9hhbkp', '6lhxfy', 'autv5r', 'nvxsrv', '6nheas', 'bag26v']
  bpx68c <class 'dict'> 7
  9hhbkp <class 'dict'> 7

Summary CSV:
(8, 28)
specimen  n_valid dv_health  t180_mean  t180_sd  t1000_mean  t1000_sd  out_180_g_mean  out_180_g_sd  in_180_g_mean  in_180_g_sd  in_dv_ms_mean  in_dv_ms_sd  t_second_ms_mean  t_second_ms_sd  e_rebound_mean  e_rebound_sd  fn_hz_mean  fn_hz_sd  zeta_pct_mean  zeta_pct_sd   H_mm  R_mm  cable_d_mm  mass_g spec  strut_d_mm  twist_deg
  6lhxfy      101   healthy   0.893078 0.004155    0.91

In [2]:
import json, os, pandas as pd, numpy as np
from scipy import stats
with open('/workspace/campaign_metrics.json') as f: cm=json.load(f)
print('campaign keys/values:')
for k,v in cm['campaign'].items():
    print(k, type(v), (v if np.isscalar(v) else (list(v)[:5] if isinstance(v,dict) else len(v))))
print('\nspecimen names', list(cm['specimens']))
first=next(iter(cm['specimens']))
print('first specimen keys', first, cm['specimens'][first].keys())
for k,v in cm['specimens'][first].items():
    print(k, type(v), (len(v) if hasattr(v,'__len__') and not isinstance(v,(str,dict)) else (list(v)[:10] if isinstance(v,dict) else v)))

campaign keys/values:
t180 <class 'dict'> ['anova_F', 'anova_p', 'means', 'spread_pct', 'median_within_cv_pct']
t1000 <class 'dict'> ['anova_F', 'anova_p', 'means', 'spread_pct', 'median_within_cv_pct']
e_rebound <class 'dict'> ['anova_F', 'anova_p', 'means', 'spread_pct', 'median_within_cv_pct']

specimen names ['6lhxfy', '6nheas', '9hhbkp', 'amdjwm', 'autv5r', 'bag26v', 'bpx68c', 'nvxsrv']
first specimen keys 6lhxfy dict_keys(['folder', 'n_captures', 'n_valid', 'invalid_signals', 'warmup_discarded', 'pauses', 'event_first', 'event_last', 'metrics', 'dv_health', 'worst_frac_fs', 'rows', 'design_params'])
folder <class 'str'> 6lhxfy 8-20-2026 101drops
n_captures <class 'int'> 101
n_valid <class 'int'> 101
invalid_signals <class 'list'> 0
warmup_discarded <class 'int'> 2
pauses <class 'list'> 0
event_first <class 'str'> 2026-08-20T19:21:51
event_last <class 'str'> 2026-08-20T20:34:07
metrics <class 'dict'> ['t180', 't1000', 'out_180_g', 'out_1000_g', 'in_180_g', 'in_raw_g', 'in_width_ms

In [3]:
from scipy import stats
D=df[['specimen','t180_mean','e_rebound_mean','out_180_g_mean','in_180_g_mean','t1000_mean','in_dv_ms_mean','t_second_ms_mean']].copy()
def assoc(x):
    r,p=stats.pearsonr(x.t180_mean,x.e_rebound_mean)
    rho,ps=stats.spearmanr(x.t180_mean,x.e_rebound_mean)
    return len(x),r,p,rho,ps
print('all 8',assoc(D))
print('exclude unreliable amdjwm',assoc(D[D.specimen!='amdjwm']))
print('\nleave-one-out all-8:')
for s in D.specimen:
    print(s, assoc(D[D.specimen!=s]))
print('\nrequested removals among reliable 7:')
R=D[D.specimen!='amdjwm']
for s in ['6lhxfy','bag26v']:
    print(s,assoc(R[R.specimen!=s]))
print('\ncorrelation matrix across specimens')
print(D.drop(columns='specimen').corr(method='pearson').round(3).to_string())
print('\nSpearman')
print(D.drop(columns='specimen').corr(method='spearman').round(3).to_string())
# per drop within specs candidate relation to input/output
rows=[]
for s,obj in cm['specimens'].items():
    rr=pd.DataFrame(obj['rows'])
    rows.append(rr.assign(specimen=s))
allrows=pd.concat(rows,ignore_index=True)
print('\nrows',allrows.shape,allrows.columns.tolist())
print(allrows.groupby('specimen')[['t180','e_rebound','out_180_g','in_180_g','in_dv_ms','t_second_ms']].size())
print('\nwithin-spec correlations in vs out, t vs input')
for s,g in allrows.groupby('specimen'):
    vals=[]
    for a,b in [('in_180_g','out_180_g'),('in_180_g','t180'),('in_dv_ms','e_rebound'),('t_second_ms','e_rebound')]:
        z=g[[a,b]].dropna(); vals.append(stats.pearsonr(z[a],z[b]).statistic if len(z)>2 else np.nan)
    print(s, [round(v,3) for v in vals])

all 8 (8, np.float64(-0.8269135402860246), np.float64(0.011339084463247466), np.float64(-0.5714285714285715), np.float64(0.13895995716070672))
exclude unreliable amdjwm (7, np.float64(-0.8436807491736352), np.float64(0.017025974288894175), np.float64(-0.39285714285714296), np.float64(0.3833168704269727))

leave-one-out all-8:
6lhxfy (7, np.float64(-0.4353048000621133), np.float64(0.3289822320833142), np.float64(-0.3571428571428572), np.float64(0.431611352038328))
6nheas (7, np.float64(-0.8819457424731951), np.float64(0.008622315584328298), np.float64(-0.39285714285714296), np.float64(0.3833168704269727))
9hhbkp (7, np.float64(-0.8416486946569692), np.float64(0.01756448454753262), np.float64(-0.7142857142857144), np.float64(0.07134356146753766))
amdjwm (7, np.float64(-0.8436807491736352), np.float64(0.017025974288894175), np.float64(-0.39285714285714296), np.float64(0.3833168704269727))
autv5r (7, np.float64(-0.8329454813602805), np.float64(0.01997943962627288), np.float64(-0.6428571428

In [4]:
# Numeric-only diagnostics needed to lock Section A: Pareto membership, uncertainty, partial-session transfer.
# Pareto fronts for minimize t180 and e_rebound
for label,x in [('all8',D),('reliable7',D[D.specimen!='amdjwm'])]:
    vals=x[['t180_mean','e_rebound_mean']].to_numpy(); names=x.specimen.tolist()
    nd=[]
    for i,v in enumerate(vals):
        dominated=any(np.all(vals[j]<=v) and np.any(vals[j]<v) for j in range(len(vals)) if j!=i)
        if not dominated: nd.append(names[i])
    print(label,'nondominated:',nd)

# session summary from numeric JSON only
with open('/workspace/partial_sessions_metrics.json') as f: pm=json.load(f)
print('\npartial session specimens:',list(pm['specimens']))
for s,o in pm['specimens'].items():
    print('\n',s,'n rows',len(o['rows']), 'metrics:')
    for k in ['t180','t1000','out_180_g','in_180_g','in_dv_ms','t_second_ms','e_rebound']:
        print(k,o['metrics'].get(k))
# Compare full and partial matched specimen means
print('\nfull matching summaries:')
for s in pm['specimens']:
    o=cm['specimens'].get(s)
    if o:
        print(s,{k:o['metrics'].get(k) for k in ['t180','t1000','out_180_g','in_180_g','in_dv_ms','t_second_ms','e_rebound']})

# Sample SEM and CV summaries
for col in ['t180','e_rebound','out_180_g','in_180_g','t1000','t_second_ms']:
    z=[]
    for s,g in allrows.groupby('specimen'):
        a=g[col].dropna(); z.append([s,a.mean(),a.std(ddof=1),a.std(ddof=1)/np.sqrt(len(a)),100*a.std(ddof=1)/a.mean()])
    print('\n',col); print(pd.DataFrame(z,columns=['spec','mean','sd','sem','cv_pct']).round(6).to_string(index=False))

all8 nondominated: ['6lhxfy', 'amdjwm', 'bpx68c']
reliable7 nondominated: ['6lhxfy', '6nheas', 'bpx68c']

partial session specimens: ['6lhxfy-s1', 'amdjwm-s1']

 6lhxfy-s1 n rows 33 metrics:
t180 {'n': 33, 'mean': 0.8943252312427908, 'sd': 0.007293078690965765, 'cv_pct': 0.8154839465762367, 'slope_pct_per_drop': -0.04340489525238494, 'slope_p': 0.0021806106776187527, 'r2': 0.26488551120703546}
t1000 {'n': 33, 'mean': 1.0289519079467375, 'sd': 0.0936112990553325, 'cv_pct': 9.097733172207517, 'slope_pct_per_drop': -0.4670761611359133, 'slope_p': 0.0032984160689737826, 'r2': 0.24644534798469456}
out_180_g {'n': 33, 'mean': 207.5628016288982, 'sd': 1.9003342557282454, 'cv_pct': 0.9155466397711549, 'slope_pct_per_drop': -0.06304273203774997, 'slope_p': 2.349537932389025e-05, 'r2': 0.44332346194518424}
in_180_g {'n': 33, 'mean': 232.08848340111416, 'sd': 0.8970585756844515, 'cv_pct': 0.38651576439235974, 'slope_pct_per_drop': -0.01961406950904209, 'slope_p': 0.0037403535693432744, 'r2': 0.24

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
locked = r'''# SECTION A — LOCKED DATA-ONLY COMMITMENT

**Scope lock:** Derived only from the numeric CSV/JSON files and the rig description supplied in the request. I had not read the campaign markdown, print-defect markdown, energy-review markdown, README, analysis scripts, BO script, or suggestion file before writing this block. This text is not to be revised; later changes must be stated as diffs.

## A1. What I would optimize

**Primary formulation:** one objective, **minimize input-adjusted CFC-180 output peak**. With these data I would operationalize that as `t180 = out_180_g / in_180_g`, while retaining `in_180_g` as a nuisance/context variable and imposing an input-severity acceptance window. A preferable future analysis, once enough independent articles exist, is a regression/analysis-of-covariance estimate of output peak at a prespecified input peak (and possibly input pulse shape/delta-v), rather than relying solely on a ratio. The ratio is currently better than raw `out_180_g` for ranking because specimen mean input spans 202.6–231.8 G and output strongly tracks input within specimens (Pearson r = 0.793–0.993). Raw output confounds design with delivered input: for example, amdjwm's low 198.6 G output occurs at the lowest 202.6 G input.

**Candidate classification:**

- **`t180`: valid primary objective, minimize**, conditional on a controlled severity window and mount checks. It directly measures peak attenuation at the specified SAE J211 channel-frequency class. It is dimensionless and normalizes much of the delivered-input variation. A peak ratio can still be biased when numerator and denominator peaks occur at different times or when pulse shape varies, so input peak and delta-v remain diagnostics.
- **`out_180_g`: valid physical endpoint but not a stand-alone objective in this campaign.** It is arguably the payload-facing quantity of greatest engineering interest. Use it as the response in an input-adjusted model or evaluate it at a fixed input severity. With uncontrolled specimen-level input variation, its raw mean is confounded.
- **`in_180_g`: constraint/context, not an objective.** It defines exposure severity. Set an acceptance window or model it as a covariate; optimizing it would optimize the rig/contact rather than protection.
- **`in_dv_ms`: constraint/context and data-quality diagnostic.** It checks impact severity and supports restitution calculations. It is not a specimen performance objective unless the specimen can physically affect the measured base pulse, in which case that feedback itself is a rig-coupling diagnostic.
- **`t1000`: diagnostic and possible safety constraint, not a second coequal objective yet.** It detects high-frequency amplification missed by CFC-180. The cross-specimen rank association with `t180` is high (Spearman rho = 0.929), but broadband values reach 1.230–1.242 for nvxsrv/bag26v. I would cap unacceptable amplification if a payload-relevant threshold is defined rather than optimize it jointly without such a threshold.
- **`t_second_ms`: diagnostic, unusable as a direct protection objective.** It is a timing observation whose event identity must be validated. It does not by itself measure energy.
- **`e_rebound = g t_second/(2 delta_v)`: diagnostic only at present, not an objective.** For ballistic flight, flight time satisfies `t = 2 v_up/g`, hence `g t/(2 delta_v) = v_up/delta_v`: a **velocity ratio**, analogous to a coefficient of restitution only if `delta_v` is the correct incident relative-speed denominator and the detected second event is landing of the same moving body. It is not an energy ratio. A corresponding specific kinetic-energy ratio would be `(g t/(2 delta_v))^2` under restrictive equal-mass/reference assumptions. The label matters physically and for thresholds/noise under squaring, though a strictly monotone square would not change deterministic ordering or the Pareto set for nonnegative values. Event identity is not established by the tabulated metrics, and amdjwm's `t_second_ms` SD is 15.70 ms versus 0.27–1.25 ms for the others, demonstrating detector failure in at least one session.
- **`fn_hz`, `zeta_pct`: opportunistic diagnostics, unusable as campaign-wide objectives.** Missing for three or more specimens/fields and likely conditional on successful mode fitting. They may explain mechanisms but cannot support a common BO response from five specimens.
- **Printed mass:** constraint/covariate, not an objective under the stated constant-solid-CAD-mass goal. Actual mass ranges 18.50–22.04 g, so the intended equality did not hold in printed articles; include measured mass in interpretation and enforce/tighten a tolerance if mass is a design requirement.
- **Derived recommendation:** report an input-adjusted CFC-180 output peak at a fixed reference severity, plus `t180` as a transparent secondary summary. Do not create an energy-absorption objective from peak ratios or hop timing without force/displacement or validated body-velocity measurements.

`t180` and `e_rebound` appear negatively associated, but the evidence does not establish two independent pieces of payload-protection physics. `e_rebound` is nearly identical to `t_second_ms` across specimen means (Pearson r = 0.998) and may reflect one compliance/restitution axis or fixture motion. I therefore would not use a Pareto formulation from these data alone.

## A2. Is multi-objective optimization warranted?

No, not with `t180` and `e_rebound` as currently measured.

Across all eight specimen means, Pearson r(`t180`, `e_rebound`) = **−0.827** (two-sided p = **0.011**), but Spearman rho = **−0.571** (p = **0.139**). Excluding amdjwm because its rebound detection is flagged unreliable gives Pearson r = **−0.844** (p = **0.017**, n = 7) but Spearman rho = **−0.393** (p = **0.383**). This conflict indicates a small-sample, leverage-sensitive relationship rather than a securely ordered trade-off.

The dependence on 6lhxfy is decisive. Among the seven reliable specimens, removing 6lhxfy changes Pearson r to **−0.429** (p = **0.395**, n = 6) and Spearman rho to **−0.029** (p = **0.957**). Removing bag26v instead gives Pearson r = **−0.849** (p = **0.033**) but Spearman rho = **−0.429** (p = **0.397**). Thus the linear anti-correlation is largely anchored by one extreme attenuator/hopper and is not robust in ranks.

For minimizing both values, the measured nondominated set is 6lhxfy, 6nheas, and bpx68c after excluding unreliable amdjwm. That empirical set shows mutual nondominance but does not prove a stable decision-relevant Pareto frontier. With n = 7 reliable articles, one article per geometry, uncertain rebound event identity, and strong single-point leverage, a two-objective BO is unsupported. Optimize CFC-180 attenuation; retain rebound timing/velocity ratio as a diagnostic until independently validated and tied to a payload requirement.

## A3. Noise for each design observation

The experimental unit for design-level optimization is an **independently printed article**, not a drop. The 99 post-warm-up drops are repeated measurements of the same article and cannot reduce print-to-print variability by `sqrt(99)`.

For an article-mean response `y`, use

`Var(y_obs | design) = sigma_print^2 + sigma_session^2 + sigma_drop^2/n_drop`,

with terms defined on a relative/log scale where practical. Given the supplied prior replication scales, my initial fixed noise for `t180` would be a **2% relative standard deviation floor**, combined with the article's drop SEM and, only if the stated 2% print scale excludes remount/session variation, a session term up to 2%:

- if the ~2% print-to-print estimate already comes from independently mounted print tests and therefore includes ordinary session/remount variability: `sigma_t180 = sqrt((0.02*y)^2 + SEM_drop^2)`;
- if print and session components were estimated separately and are independent: `sigma_t180 = sqrt((0.02*y)^2 + sigma_session^2 + SEM_drop^2)`, with `sigma_session` estimated from matched re-mounts rather than automatically taking the maximum observed 2% shift.

At `t180 ~ 1`, the first formula gives SD ~**0.020**, versus the observed per-drop SEM ~0.00017–0.00052. The supplied ~0.0004 treatment is therefore ~50-fold too small in SD and ~2,500-fold too small in variance for predicting a new print.

No print-to-print replication scale is supplied for `e_rebound`, so I would **not invent one and would not hand it to the GP as near-zero fixed noise**. If it were retained after event validation, obtain independent-print replicates and estimate a log-scale article variance. Until then, a transparent provisional sensitivity analysis could use at least a 2% relative floor, `sqrt((0.02*e)^2 + SEM_drop^2)`, but that assumes transfer of the t180 print CV and is not evidentially justified. The correct data-only decision is to keep rebound diagnostic rather than pretend its design-level noise is known.

## A4. Committed answer

1. **Optimize one endpoint:** minimize input-adjusted CFC-180 output peak, using `t180` now and moving toward output-at-fixed-input when replication supports it. Treat input peak/delta-v as exposure controls; constrain gross CFC-1000 amplification; keep rebound and modal fits diagnostic.
2. **Do not run `t180` + `e_rebound` qNEHVI from these eight articles.** The rebound metric is a conditional velocity ratio, not energy; event identity is unvalidated; rank evidence is weak; and the association collapses without 6lhxfy.
3. **Use article-level noise.** For `t180`, start with `sigma = sqrt((0.02*y)^2 + SEM_drop^2)` if the print estimate includes mounting/session effects. Do not use per-drop SEM alone. Do not assign a precise design-level rebound noise without independent-print data.
'''
outdir=Path('/workspace/edison-trajectories/bo-objectives'); outdir.mkdir(parents=True,exist_ok=True)
path=outdir/'SECTION_A_LOCKED.md'; path.write_text(locked)
h=hashlib.sha256(path.read_bytes()).hexdigest()
stamp=datetime.now(timezone.utc).isoformat()
manifest=f"locked_at_utc: {stamp}\nsha256: {h}\nfile: {path}\n"
(outdir/'SECTION_A_LOCKED.sha256.txt').write_text(manifest)
print(manifest)
print('chars',len(locked))

locked_at_utc: 2026-08-21T05:55:22.944427+00:00
sha256: c1f43a8a1f913e0b8390d291b1a27ef780c49555131d8d3a915dde3b2d5aea64
file: /workspace/edison-trajectories/bo-objectives/SECTION_A_LOCKED.md

chars 9733


In [6]:
# Post-lock quantitative audit: session transfer, detector fragility, parameter correlations, and noise scales.
from scipy import stats
# matched full vs partial sessions, accounting suffix
for partial_name, full_name in [('6lhxfy-s1','6lhxfy'),('amdjwm-s1','amdjwm')]:
    a=pm['specimens'][partial_name]['metrics']; b=cm['specimens'][full_name]['metrics']
    print('\n',full_name)
    for k in ['t180','t1000','e_rebound','t_second_ms','in_dv_ms']:
        ma=a[k]['mean']; mb=b[k]['mean']; print(k,ma,mb,'pct change',100*(mb-ma)/ma)
# Detector quality proxy second_rel_db and timing distributions / missingness
print('\nsecond-event diagnostics')
for s,g in allrows.groupby('specimen'):
    q=g[['t_second_ms','e_rebound','second_rel_db']].describe(percentiles=[.01,.05,.5,.95,.99])
    ts=g.t_second_ms
    print(s,'n',ts.notna().sum(),'range',ts.min(),ts.max(),'unique rounded',len(np.unique(np.round(ts.dropna(),3))),
          'second dB mean/sd/range',g.second_rel_db.mean(),g.second_rel_db.std(),(g.second_rel_db.min(),g.second_rel_db.max()))
# Design parameter association (exploratory only)
train=df[df.specimen!='amdjwm'].copy()
pars=['R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm','mass_g']
print('\nSpearman parameter associations n=7')
for y in ['t180_mean','e_rebound_mean','t1000_mean']:
    print(y,{p:round(stats.spearmanr(train[p],train[y]).statistic,3) for p in pars})
# Suggested bounds exact
sugg=pd.read_csv('/workspace/t3-prism-bo-suggestions-round1.csv')
for p,lo,hi in [('R_mm',25,40),('H_mm',60,110),('twist_deg',40,80),('strut_d_mm',6,12),('cable_d_mm',3,5.5)]:
    print(p,'lo',np.isclose(sugg[p],lo).sum(),'hi',np.isclose(sugg[p],hi).sum())
# empirical design-level CV supplied in defect document: compute implications SD vs spread
means=np.array([1.0432,1.0374,1.0315,1.0336,1.0231])
print('\ndefect means: mean sd CV range/mean',means.mean(),means.std(ddof=1),means.std(ddof=1)/means.mean(),(means.max()-means.min())/means.mean())
# If 0.72% T SD + per-drop SEM, range of abs sigma
cv=.0072
print('campaign T abs design sd',[(s,round(cv*y,6)) for s,y in zip(df.specimen,df.t180_mean)])


 6lhxfy
t180 0.8943252312427908 0.8930777877843858 pct change -0.1394843189956183
t1000 1.0289519079467375 0.9126730324377276 pct change -11.300710423001519
e_rebound 0.04726086255216924 0.05037622154824482 pct change 6.5918369404212
t_second_ms 53.126060606061884 55.178585858587184 pct change 3.863499813669788
in_dv_ms 5.511977219296571 5.3720484060381635 pct change -2.538631922652703

 amdjwm
t180 0.9861595780405985 0.9804953200642735 pct change -0.5743753954688846
t1000 1.1595299708958333 1.0070155415571158 pct change -13.153125246162238
e_rebound 0.029315294541114888 0.029617813670512614 pct change 1.031949820505611
t_second_ms 29.969411764706607 31.772929292930055 pct change 6.017860952303895
in_dv_ms 5.012704212621296 5.26116704531628 pct change 4.956662554901783

second-event diagnostics
6lhxfy n 99 range 54.38000000000132 57.00000000000138 unique rounded 45 second dB mean/sd/range -9.361037579158836 0.7965985318334476 (np.float64(-10.790036681626678), np.float64(-7.82428805276

In [7]:
# Bootstrap uncertainty and covariate ranking checks for the report.
import numpy as np, pandas as pd
from scipy import stats
rng=np.random.default_rng(20260821)
R=D[D.specimen!='amdjwm'].reset_index(drop=True)
def boot_corr(x,nboot=200000):
    a=x.t180_mean.to_numpy(); b=x.e_rebound_mean.to_numpy(); n=len(x)
    inds=rng.integers(0,n,size=(nboot,n))
    # discard resamples with zero variance
    aa=a[inds]; bb=b[inds]
    ac=aa-aa.mean(1,keepdims=True); bc=bb-bb.mean(1,keepdims=True)
    den=np.sqrt((ac*ac).sum(1)*(bc*bc).sum(1)); rr=(ac*bc).sum(1)/den
    rr=rr[np.isfinite(rr)]
    return np.quantile(rr,[.025,.5,.975]), len(rr)
print('bootstrap Pearson reliable7',boot_corr(R))
# Partial correlation t180/e controlling design-level input peak or dv separately (descriptive n=7)
def partial_corr(data,x,y,z):
    rx=stats.linregress(data[z],data[x]); ry=stats.linregress(data[z],data[y])
    ex=data[x]-(rx.intercept+rx.slope*data[z]); ey=data[y]-(ry.intercept+ry.slope*data[z])
    return stats.pearsonr(ex,ey)
for z in ['in_180_g_mean','in_dv_ms_mean','mass_g']:
    print('partial controlling',z,partial_corr(df[df.specimen!='amdjwm'],'t180_mean','e_rebound_mean',z))
# Regression raw output on input: ranking residuals is extremely underpowered but descriptive
x=df[['specimen','out_180_g_mean','in_180_g_mean','t180_mean']].copy()
fit=stats.linregress(x.in_180_g_mean,x.out_180_g_mean)
x['out_resid']=x.out_180_g_mean-(fit.intercept+fit.slope*x.in_180_g_mean)
print('out on input',fit)
print(x.sort_values('out_resid').to_string(index=False))
print('Spearman rank t180 vs raw output',stats.spearmanr(df.t180_mean,df.out_180_g_mean))
print('Spearman t180 vs input-adjusted residual',stats.spearmanr(x.t180_mean,x.out_resid))

bootstrap Pearson reliable7 (array([-0.98197661, -0.8573617 ,  0.76205035]), 200000)
partial controlling in_180_g_mean PearsonRResult(statistic=np.float64(-0.8276184321277824), pvalue=np.float64(0.021545147134024493))
partial controlling in_dv_ms_mean PearsonRResult(statistic=np.float64(-0.8842402677084455), pvalue=np.float64(0.008219956885171928))
partial controlling mass_g PearsonRResult(statistic=np.float64(-0.8167500180502986), pvalue=np.float64(0.024948291605062595))
out on input LinregressResult(slope=np.float64(0.875695732939261), intercept=np.float64(27.9154647339073), rvalue=np.float64(0.6737504932781615), pvalue=np.float64(0.0669577616344686), stderr=np.float64(0.3921017702162992), intercept_stderr=np.float64(85.85639399987728))
specimen  out_180_g_mean  in_180_g_mean  t180_mean  out_resid
  6lhxfy      200.040828     224.012459   0.893078 -24.041391
  amdjwm      198.638701     202.594229   0.980495  -6.687666
  6nheas      226.340773     227.022670   0.997008  -0.377475
  b

In [8]:
from e14c.web import web_search
queries=[
 'SAASBO sparse axis-aligned subspaces Bayesian optimization Eriksson Jankowiak 2021 arxiv',
 'qNEHVI noisy expected hypervolume improvement Daulton Balandat Bakshy 2021 paper',
 'heteroskedastic noise Bayesian optimization replication experimental design paper',
 'SAE J211 CFC 180 filtering impact instrumentation standard',
 'shock response spectrum standard practice transient shock ISO 18431-4',
 'coefficient restitution time between bounces ballistic flight paper Bernstein 1977'
]
results=[]
for q in queries:
    r=await web_search(q, category='scholarly', num_results=5)
    results.append(r)
    print('\nQUERY',q)
    for x in r[:3]: print(x.get('title'),x.get('url'),x.get('publishedDate'))


QUERY SAASBO sparse axis-aligned subspaces Bayesian optimization Eriksson Jankowiak 2021 arxiv
High-Dimensional Bayesian Optimization with Sparse Axis-Aligned Subspaces https://doi.org/10.48550/arxiv.2103.00349 2021-02-27T00:00:00.000Z
207 II-D3 High-Dimensional Bayesian Optimization with Sparse Axis-Aligned Subspaces https://doi.org/10.48448/w6z4-2992 2021-07-17T00:00:00.000Z
Computationally Efficient High-Dimensional Bayesian Optimization via Variable Selection https://doi.org/10.48550/arxiv.2109.09264 2021-09-20T00:00:00.000Z



QUERY qNEHVI noisy expected hypervolume improvement Daulton Balandat Bakshy 2021 paper
Parallel Bayesian Optimization of Multiple Noisy Objectives with Expected Hypervolume Improvement https://doi.org/10.48550/arxiv.2105.08195 2021-05-17T00:00:00.000Z
Parallel Bayesian Optimization of Multiple Noisy Objectives with Expected Hypervolume Improvement | Semantic Scholar https://www.semanticscholar.org/reader/8f5562ead9861744a1192c1bef69283e25200aa8 2021-05-17T00:00:00.000Z
Appendix to: https://papers.neurips.cc/paper_files/paper/2021/file/11704817e347269b7254e744b5e22dac-Supplemental.pdf None



QUERY heteroskedastic noise Bayesian optimization replication experimental design paper
Replication or Exploration? Sequential Design for Stochastic Simulation Experiments https://doi.org/10.1080/00401706.2018.1469433 2018-05-09T00:00:00.000Z
Achieving Robustness to Aleatoric Uncertainty with Heteroscedastic Bayesian Optimisation https://doi.org/10.1088/2632-2153/ac298c 2021-09-23T00:00:00.000Z
hetGP: Heteroskedastic Gaussian Process Modeling and Sequential Design in R https://doi.org/10.18637/jss.v098.i13 2021-01-01T00:00:00.000Z



QUERY SAE J211 CFC 180 filtering impact instrumentation standard
Instrumentation for Impact Test Part 1 - Electronic Instrumentation https://doi.org/10.4271/j211/1_202208 2022-08-19T00:00:00.000Z
Digital Filtering for J211 Requirements using a Fast Fourier Transform Based Filter https://doi.org/10.4271/2002-01-0796 2002-03-04T00:00:00.000Z
Performance Specifications for Anthropomorphic Test Device Transducers https://doi.org/10.4271/j2570_202506 2025-06-30T00:00:00.000Z



QUERY shock response spectrum standard practice transient shock ISO 18431-4
Calculation of a Shock Response Spectra – DOAJ https://doaj.org/article/2905cf12ac404a71a4eb7e7fced0463c None
Quantifying Mitigation Characteristics of Shock Isolation Seats in a Wave Impact Environment https://doi.org/10.21236/ada622526 2015-01-01T00:00:00.000Z
Mechanical Shock https://onlinelibrary.wiley.com/doi/10.1002/9781118931127.ch3 None



QUERY coefficient restitution time between bounces ballistic flight paper Bernstein 1977
Listening to the coefficient of restitution https://doi.org/10.1119/1.10904 1977-01-01T00:00:00.000Z
Listening to the coefficient of restitution and the gravitational acceleration of a bouncing ball https://doi.org/10.1119/1.1524166 2003-04-10T00:00:00.000Z
Using Microsoft Windows To Compare The Energy Dissipated By Old And New Tennis Balls https://doi.org/10.18260/1-2--15818 2020-09-03T00:00:00.000Z


In [9]:
# Parse TP4 series tables numerically and compare their independent peak/delta-v summaries with campaign metrics.
import csv, glob, re, os, pandas as pd, numpy as np
recs=[]
for fp in glob.glob('/workspace/series-table_*.csv'):
    lines=open(fp,encoding='utf-8-sig').read().splitlines()
    specimen=os.path.basename(fp).split('_')[1]
    # data starts line 7; each row: trigger,event,time,order,(G,duration,dv)*4
    vals=[]
    for line in lines[6:]:
        row=next(csv.reader([line]))
        try:
            nums=[float(row[4+3*j].strip()) for j in range(4)]
            dvs=[float(row[6+3*j].strip()) for j in range(4)]
            vals.append(nums+dvs)
        except Exception: pass
    a=np.asarray(vals,float)
    if len(a):
        # TOP resultant proxy from independently reported per-axis absolute peaks; peaks need not be simultaneous
        top_proxy=np.sqrt((a[:,:3]**2).sum(1)); ch5=np.abs(a[:,3]); ch5dv=np.abs(a[:,7])*0.0254
        recs.append(dict(specimen=specimen,n=len(a),tp4_top_proxy_mean=top_proxy.mean(),tp4_ch5_peak_mean=ch5.mean(),
                         tp4_peak_ratio_proxy=(top_proxy/ch5).mean(),tp4_ch5_dv_ms=ch5dv.mean()))
tp4=pd.DataFrame(recs)
print(tp4.to_string(index=False))
# match full sessions only
full=tp4[~tp4.specimen.str.endswith('-s1')].merge(df,left_on='specimen',right_on='specimen')
for a,b in [('tp4_ch5_peak_mean','in_180_g_mean'),('tp4_peak_ratio_proxy','t180_mean'),('tp4_ch5_dv_ms','in_dv_ms_mean')]:
    print(a,b,'Pearson',stats.pearsonr(full[a],full[b]),'Spearman',stats.spearmanr(full[a],full[b]))
print(full[['specimen','tp4_ch5_peak_mean','in_180_g_mean','tp4_peak_ratio_proxy','t180_mean','tp4_ch5_dv_ms','in_dv_ms_mean']].to_string(index=False))
# e session difference variance estimate
rel=np.array([6.59183694,1.03194982])/100
sig_session=np.sqrt(np.sum(rel**2)/(2*len(rel)))
print('e provisional per-session relative sigma from two paired differences',sig_session, 'RMS diff',np.sqrt(np.mean(rel**2)))
# exact CI assuming zero mean normal paired differences, df=2
from scipy.stats import chi2
ss=np.sum(rel**2)/2 # sum d² /2 = sum implied session squares
print('rough 95% CI sigma session',np.sqrt(ss/chi2.ppf(.975,2)),np.sqrt(ss/chi2.ppf(.025,2)))

 specimen   n  tp4_top_proxy_mean  tp4_ch5_peak_mean  tp4_peak_ratio_proxy  tp4_ch5_dv_ms
   amdjwm 101          269.577041         241.694950              1.115968       5.317656
   bpx68c 101          505.790178         252.544950              2.004014       5.386264
   nvxsrv 101          619.421532         290.200000              2.134663       5.397284
6lhxfy-s1  35          559.320566         306.851714              1.800788       5.396128
amdjwm-s1  87          423.767630         237.404828              1.789334       5.137643
   9hhbkp 101          332.747771         295.739505              1.125074       5.453654
   6lhxfy 101          459.588696         284.623366              1.619331       5.483269
   bag26v 101          407.250691         274.692178              1.483779       5.125416
   6nheas 101          315.563879         283.780792              1.112065       5.406403
   autv5r 103          421.484042         295.674951              1.428203       5.388664
tp4_ch5_pe

In [10]:
from pathlib import Path
report = r'''# Adversarial review of the T3-prism Bayesian-optimization objectives

**Review date:** 2026-08-21  
**Decision:** **Do not print the nine round-2 suggestions as-is.** Refit the optimizer with one primary objective and article-level noise, then regenerate the batch. Before deciding whether rebound belongs in any future optimization, run a short synchronized-video restrained/unrestrained test. The present batch was generated from a mislabeled, physically unvalidated second objective and a materially understated noise model.

## What changes the next print

1. **Drop `e_rebound` from qNEHVI.** It is a velocity ratio only under a restrictive ballistic interpretation, not an energy ratio. The bundle does not establish what body is in flight, that the carriage is stationary during flight, or that the detected second event is consistently a landing. Its detector fails conspicuously for `amdjwm` and is weak for other sessions.
2. **Use one primary response: minimize CFC-180 filtered peak ratio `t180`, with input severity controlled and recorded.** This is defensible as a screening endpoint, not as “energy absorbed.” It distinguishes these articles strongly, but it remains vulnerable to geometry-dependent mount coupling.
3. **Do not pass per-drop SEM as design noise.** The optimizer predicts a new printed article. Drops are repeated measurements on one article. The attached five-print study estimates a between-article-plus-mount CV of **0.72%**, not 2%; **1.95% is the observed five-print range**, not a standard deviation. Use a `t180` observation SD of approximately `sqrt((0.0072*y)^2 + SEM_drop^2)`, about **0.0064–0.0076** here. This is still 14–44 times the per-drop SEM in SD, depending on specimen.
4. **The current boundary collapse is not evidence that the corner is optimal.** Eight of nine suggestions set strut diameter to 12 mm, seven set height to 60 mm, and most other coordinates also hit bounds. With seven completed points in five dimensions, a batch of nine, fixed near-zero observation noise, and an extreme `6lhxfy` observation, SAASBO can interpret article/mount variation and one-point leverage as a steep deterministic surface. qNEHVI then spends the batch exploiting/extending a fragile inferred front. The collapse is consistent with the bad noise specification, though not uniquely caused by it.
5. **Reallocate drops to prints.** For each selected geometry, use at least **3 independently printed articles × 10–12 stabilized drops/article**, randomized in blocks with a reference article before/after each block. Ninety-nine analyzed drops on one print estimate that article very precisely but add almost no information about the next print.

---

# A. Locked data-only commitment

> **Provenance:** This block was written before reading the campaign markdown, print-defect markdown, energy-review markdown, README, analysis scripts, BO script, or suggestion file. It was saved at `edison-trajectories/bo-objectives/SECTION_A_LOCKED.md` on 2026-08-21T05:55:22Z with SHA-256 `c1f43a8a1f913e0b8390d291b1a27ef780c49555131d8d3a915dde3b2d5aea64`. The wording below is condensed for this report; the locked file retains the full original text.

## A1. Data-only objective choice

**Committed choice:** one objective, minimize input-adjusted CFC-180 output peak. Use `t180 = out_180_g/in_180_g` now, with `in_180_g` and `in_dv_ms` as exposure controls. With future independent-article replication, prefer output peak predicted at a prespecified input peak and pulse severity over an unqualified ratio.

| Candidate | Data-only classification | Reason |
|---|---|---|
| `t180` | **Objective, minimize** | Directly represents CFC-180 peak attenuation and removes much input-level variation. It is a filtered peak ratio, not a frequency-response transmissibility. |
| `out_180_g` | Endpoint requiring input adjustment | This is payload-facing, but raw specimen means are confounded by input means of 202.6–231.8 G. Within specimens, input/output peak Pearson correlations are 0.793–0.993. |
| `in_180_g` | Constraint/covariate | Exposure severity, not performance. |
| `in_dv_ms` | Constraint/diagnostic | Exposure and rig-health descriptor. |
| `t1000` | Diagnostic or threshold constraint | Flags broadband amplification; values reach 1.230–1.242. It is too session-sensitive to be coequal. |
| `t_second_ms` | Diagnostic | Timing alone is not damage or energy. Event identity is unvalidated. |
| `e_rebound` | Diagnostic only | Under ballistic assumptions it is `v_up/delta_v`, a velocity ratio. It is not an energy ratio. |
| `fn_hz`, `zeta_pct` | Opportunistic diagnostics | Fits are unavailable or unusable for several articles. |
| measured mass | Constraint/covariate | Printed mass varies 18.50–22.04 g despite constant solid-CAD mass. |

For a stationary landing surface and a body launched vertically at speed `v_up`, flight time is

`T_flight = 2 v_up/g`, so `g T_flight/(2 delta_v) = v_up/delta_v`.

An energy ratio would require squaring the velocity ratio and specifying compatible masses and reference states. Squaring a nonnegative objective is monotone, so it preserves deterministic ordering and Pareto membership, but it changes units, thresholds, uncertainty propagation, and interpretation.

## A2. Data-only assessment of a Pareto formulation

**Committed answer: no.** Across all eight specimen means, Pearson `r = -0.827` (`p = 0.011`), but Spearman `rho = -0.571` (`p = 0.139`). Excluding unreliable `amdjwm`, Pearson `r = -0.844` (`p = 0.017`, `n = 7`) while Spearman `rho = -0.393` (`p = 0.383`).

The dependence on one point is decisive:

| Reliable observations used | Pearson r | p | Spearman rho | p |
|---|---:|---:|---:|---:|
| all 7 | -0.844 | 0.017 | -0.393 | 0.383 |
| excluding `6lhxfy` | -0.429 | 0.395 | -0.029 | 0.957 |
| excluding `bag26v` | -0.849 | 0.033 | -0.429 | 0.397 |

A specimen-bootstrap 95% interval for Pearson correlation over the seven reliable points is **-0.982 to +0.762**, showing how little the sample fixes the population association. The empirical nondominated set for minimizing both metrics is `6lhxfy`, `6nheas`, and `bpx68c`, but three mutually nondominated observed articles do not establish a stable physical frontier.

## A3. Data-only noise commitment

**Committed replication unit:** independently printed article. The proposed formula was

`Var(y_article | design) = sigma_print^2 + sigma_session^2 + sigma_drop^2/n_drop`.

Based only on the prompt’s statement that printing moved `T` by ~2%, I provisionally committed to a 2% relative SD floor for `t180`, with no invented floor for rebound. I explicitly rejected per-drop SEM alone.

## A4. Locked answer

1. Optimize `t180` as the current input-normalized CFC-180 peak endpoint; develop output-at-fixed-input when replicated data support it.
2. Do not run `t180` plus `e_rebound` qNEHVI from these observations.
3. Use article-level, not drop-level, uncertainty.

## Explicit diff after reading the team documents

- **Change:** replace the provisional **2% SD** for `t180` with **0.72% relative SD** as the best bundle-based starting value.  
  **Why:** the print-defect study reports five article means with between-article CV 0.72% and a worst-to-best spread of 1.95%. The campaign text repeatedly calls the latter a “~2% floor,” but spread is not SD. The estimate also includes mount/order/session confounding and is therefore an upper bound on pure print variation, not a clean print-only component.
- **No change:** reject `e_rebound` as an optimization objective and reject per-drop SEM as design noise. Reading the documents strengthens both objections. The energy review itself states that the restrained-versus-unrestrained experiment remains open.
- **Refinement:** raw `out_180_g` should not replace `t180` without conditioning. Across the eight articles, raw-output rank versus `t180` is only Spearman `rho = 0.595` (`p = 0.120`); a simple output-on-input residual gives exactly the `t180` ranking here (`rho = 1.0`), but this is an eight-point descriptive fit, not a validated calibration model.

---

# B. `e_rebound` fails as an objective

## B1. Dimensional and physical audit

`e_rebound = g t_second/(2 delta_v)` is dimensionless. Under ideal ballistic flight, it estimates a **launch-velocity ratio**. Calling it a “rebound energy ratio” is wrong. If the numerator and denominator refer to the same mass and compatible states, the corresponding kinetic-energy ratio is `e_rebound^2`, not `e_rebound`.

There are additional assumptions hidden in the equation:

1. the detected interval starts at separation and ends at re-contact;
2. the same rigid body is in free flight throughout;
3. its launch and landing elevations are equal;
4. aerodynamic effects are negligible;
5. the landing surface is stationary or its motion is modeled;
6. `delta_v` is the relevant incident relative velocity.

The rig violates or has not demonstrated several of these. `in_dv_ms` is a full-pulse base integral and includes arrest plus rebound contributions; it is not automatically the incident specimen/plate relative velocity. The top sensor is attached to one vertex of a deformable tensegrity, not its center of mass or a known payload mass.

**Effect on BO:** replacing positive `e` with `e^2` would preserve a noiseless Pareto set, but not posterior inference. By the delta method, `Var(e^2) ≈ 4 e^2 Var(e)`, distributions become skewed near zero, and any fixed absolute SEM changes scale. Physical thresholds also change. The mislabeled quantity therefore affects more than prose once uncertainty or engineering limits enter.

## B2. What is hopping?

The bundle cannot establish it.

- The campaign records show a repeatable second-event time for most articles, but no synchronized video is included.
- The energy-review document asserts that the top vertex separates and lands, then says the restrained/unrestrained check is still scheduled. That is an inference followed by an admission that the discriminating test has not been run.
- No channel-resolved analysis demonstrates a top-only landing impulse absent from CH5, nor rules out carriage/mat restitution, plate motion, a rocking/recontact event, or a ringdown lobe.
- The carriage does rebound: the campaign analysis notes that full-pulse delta-v contains arrival plus rebound speed. A ballistic calculation relative to a moving plate is not the stated formula.

The TP4 series tables independently report event-level peaks, durations, and delta-v but do not report the late second event. They therefore validate the main pulse only, not the hop interpretation.

Even if the whole specimen leaves the plate, `0.5 m v_up^2` is energy of the specimen or some effective moving mass, not “energy returned to the payload.” The top accelerometer has negligible payload mass, and vertex deformation means its kinematics need not equal center-of-mass motion.

## B3. Direction of goodness

Both stories are plausible:

- **Minimize rebound:** a second impact can damage a payload, and retained elastic energy can create repeated shocks.
- **Do not penalize rebound automatically:** elastic storage and delayed return may lower the first transmitted peak. `6lhxfy` has the lowest `t180` (0.8931) and longest detected delay (55.18 ms). Penalizing its timing-derived velocity ratio can oppose the mechanism that reduces the primary peak.

For this rig, neither story makes `e_rebound` a valid objective because the metric omits the quantity needed to decide: the **amplitude, duration, direction, and payload response of the second impact**. A long soft hop and a short hard recontact can order differently by damage risk. If repeated impact is unacceptable, constrain the second-event CFC-180 peak or a payload-relevant shock-response-spectrum (SRS) ordinate after validating event identity. Do not minimize flight time as a surrogate.

## B4. Detector fragility

`amdjwm` is not a small anomaly. Its full session has `t_second` mean 31.77 ms, SD 15.70 ms, and range 22.06–69.96 ms; its `e_rebound` CV is 49.5%. The analysis document says the picker alternates between a landing candidate and a ringdown lobe.

Other warning signs:

- `nvxsrv`: `t_second` SD 1.25 ms and range 27.58–32.00 ms, wider than the 0.27–0.89 ms SD of most articles.
- `6lhxfy`: the detected second feature averages about **-9.36 dB** relative to the algorithm’s reference; `amdjwm` averages **-12.18 dB**. Soft events approach ringdown structure and are inherently harder to pick.
- Cross-session `e_rebound` shifts are **+6.59%** for `6lhxfy` and **+1.03%** for `amdjwm`, despite `t180` shifts of only -0.14% and -0.57%. With only two pairs, a rough independent-session relative SD estimate for rebound is 3.34%, with enormous uncertainty.

A fixed picker that always returns a value does not make the estimand observable. Soft, split, rocking, or multimodal recontacts require an explicit confidence/quality output and censoring model. Until synchronized video validates the detector across geometries, `e_rebound` is diagnostic.

---

# C. The claimed Pareto trade-off is not established

## C1. Association and leverage

The exact correlations are in Section A2. The headline linear anti-correlation is real for these observed means, but “genuinely trade off” is too strong because:

- rank correlation is nonsignificant;
- removing `6lhxfy` nearly eliminates rank association (`rho = -0.029`);
- one rebound value is unreliable;
- there are only seven training articles;
- uncertainty is at the article level, not the ~99-drop level.

The statement “best attenuator hops hardest” is one observation, not evidence of a general frontier.

## C2. Frontier versus one compliance axis

Two hypotheses remain:

1. **Decision-relevant conflict:** design variables independently control primary peak and validated rebound damage. Round 2 should populate distinct regions of a stable front, and replicated articles should preserve nondominance.
2. **One physical axis:** a compliant path shifts energy in time, reducing the first peak while increasing delayed motion. Then `t180` and hop timing are two projections of the same mechanism, and whether there is a conflict depends on a separate damage requirement for the delayed event.

Existing data do not discriminate these. Exploratory Spearman associations among the seven mapped articles are `rho(t180,H) = 0.929` and `rho(t180,strut diameter) = -0.750`, whereas rebound associations are weaker (`rho = -0.393` and `+0.321`, respectively). That could indicate partially different controls, detector noise, or small-sample confounding. Five variables with seven points cannot separate them.

A useful discriminating round would replicate designs that vary one suspected compliance control at a time, then measure main and second-event payload response directly. The proposed boundary-heavy batch is not such a test.

## C3. qNEHVI behavior and the collapsed batch

Noisy expected hypervolume improvement is designed for noisy multi-objective observations, but it can only honor the uncertainty supplied to it. Passing SD ~0.0004 for `t180` tells the model that differences of a few thousandths are highly reliable design effects. They are not reliable for a newly printed article.

In a sparse axis-aligned subspace Gaussian process (SAASBO), shrinkage encourages a few active dimensions. With seven completed points in five dimensions, an extreme low-`t180` point at relatively thick struts, short height, thin cables, and high twist can induce a simple steep story. qNEHVI then searches for hypervolume gains along that inferred surface while simultaneously filling a batch of nine. Boundaries are natural targets when a fitted trend has no observed turnover.

This is consistent with the output: 8/9 at maximum strut diameter, 7/9 at minimum height, 5/9 at each radius extreme, 8/9 at a twist extreme, and all 9 at a cable-diameter extreme. It does **not** prove that noise misspecification alone caused collapse. Search-space projection, sparse data, large batch size, model priors, and legitimate boundary optima can all contribute. The correct test is to refit with article-level noise and one objective and compare posterior diagnostics and suggestions across seeds.

---

# D. The noise model is the clearest failure

## D1. Consequences of per-drop SEM

The GP observation is one article mean. Per-drop SEM describes uncertainty in that article’s session mean, not variability of a future article made from the same geometry.

For campaign `t180`, drop SEM ranges about 0.00017–0.00052. The five-print study gives between-article-plus-mount SD `0.00744` at mean `1.03376`, CV **0.719%**. At campaign responses this implies SD **0.00643–0.00764**. The current likelihood is therefore too narrow by roughly **14–44 times in SD** and **~200–1,900 times in variance**, not a universal 50×. The “50×” comparison is only obtained by incorrectly treating the observed ~2% range as an SD.

Likely consequences are interpolation of print/mount lottery as geometry, exaggerated confidence in Pareto membership, spuriously short length scales or active-dimension selection, and aggressive exploitation. With `n = 7`, none of those diagnostics is stable.

## D2. What the print study actually supports

The five nominally identical article means are 1.0432, 1.0374, 1.0315, 1.0336, and 1.0231:

- mean: 1.03376;
- sample SD: 0.00744;
- CV: **0.719%**;
- range/mean: **1.944%**.

The 0.72% is not a clean print-only SD. Defect grade is confounded with test order, day, and a felt adjustment; accelerometer seating changes between specimens. The document correctly calls it an upper bound on pure print scatter. The study also used a different absorber arrangement and 20 ms records, which weakens transfer.

There is **no five-print estimate for `e_rebound`** because those 20 ms captures do not contain the later landing. The two matched session reruns bound only combined session/remount/detector effects for the same articles. They show far larger proportional movement in rebound than in `t180`, but two pairs cannot estimate a trustworthy variance component.

## D3. Concrete round-2 noise treatment

For `t180`, use a relative/log-scale model if available. A defensible fixed-noise approximation is

`sigma_i = sqrt((0.0072*y_i)^2 + SEM_drop,i^2)`.

This treats 0.72% as total article-level measurement/realization noise and avoids double-counting session effects already embedded in the five-print study. Run sensitivity fits at 0.5%, 0.72%, 1%, and 2% because the estimate is based on five confounded articles and a different test arrangement.

With fewer than ten points, estimating separate heteroskedastic print, session, and drop components inside the Ax Service model is not identifiable. Letting a homoskedastic GP infer all noise from seven points is also fragile. For this round, fixed empirically grounded noise plus sensitivity analysis is more defensible. Once replicated articles exist, fit a hierarchical model:

`y_design,article,session,drop = f(design) + article(design) + session(article) + drop error`,

and pass the posterior uncertainty for a new article or use a replicate-aware stochastic-kriging/heteroskedastic GP workflow.

If rebound is retained only for sensitivity, a provisional floor cannot be called print noise. The two session pairs suggest ~3.3% relative session SD, and detector failures argue for at least **5% relative SD plus explicit invalid/censored observations**. This is a conservative engineering sensitivity choice, not an estimate of print-to-print rebound variance. The preferred action is not to optimize it.

## D4. Better allocation

The 101-drop allocation is inefficient for design learning. Recommended for the same campaign stage:

- **3 independently printed articles per geometry**;
- **10–12 analyzed drops per article** after two warm-ups;
- randomized article order in blocks;
- the same reference article at block start/end;
- no mat adjustment within a block;
- fresh seating documented for each article.

Three articles × 12 analyzed drops gives 36 observations per geometry while changing the true design replication from one to three. If nine total print slots are fixed, do not spend all nine on nine new geometries. A practical compromise is **five new geometries plus four replicate prints**, including two additional `6lhxfy` articles and replication at a near-neutral/reference geometry. Exact allocation should be chosen after the corrected one-objective acquisition is generated.

---

# E. `t180` survives, but only as a screening metric

## E1. What the new campaign settles

The new evidence defeats the narrow claim that CFC-180 peak ratio cannot discriminate these tested articles. It spans 0.8931–1.0616, a 16.8% campaign spread, with within-article CV 0.17–0.48%. `6lhxfy` repeats across sessions to 0.14%; `amdjwm` to 0.57%. CFC-1000 does not transfer, changing -11.3% and -13.2% in the two reruns.

It does **not** establish that the difference is intrinsic structural attenuation. A mount or key-seat artifact can be stable for one geometry and reproducible after remounting if geometry controls contact area, wax thickness, orientation, local stiffness, or sensor alignment. Actual printed mass also covaries with design because constant solid mass did not yield constant printed mass.

Discriminating tests are:

1. independently printed replicate articles with randomized order;
2. repeated blind remounts per article;
3. an alternate top-sensor mounting or a small rigid payload plate with known mass;
4. synchronized video/relative-displacement measurement;
5. bench modal/transfer tests independent of the drop fixture.

The tail re-baseline and TP4 agreement support processing validity for the main pulse, not structural attribution.

## E2. Alternative metrics and ranking

What can be tested from the bundle:

- Raw `out_180_g` does **not** preserve ranking (`rho = 0.595` versus `t180`); it is input-confounded.
- A simple eight-article regression residual of output on input preserves the `t180` ranking exactly, but is underpowered and partly algebraic because `t180` is already the ratio.
- `t1000` is strongly rank-associated with `t180` (`rho = 0.929`) but changes the middle ordering and fails session transfer.

What cannot be tested from the attached derived files:

- SRS ratios at payload frequencies;
- band-limited frequency-domain transmission;
- output/input impulse ratios;
- simultaneous-peak or time-domain transfer measures.

Those require raw channel time histories, which are not in this bundle. The TP4 series tables contain independent per-axis event extrema, but axis peaks are not simultaneous and cannot reconstruct top-resultant CFC-180 peaks or SRS. Therefore no honest claim of ranking invariance across SRS, impulse, and band-limited alternatives is possible here.

SAE J211 channel-frequency-class filtering supports consistent impact-channel processing; it does not make a ratio of nonsimultaneous maxima a transfer function. SRS is standard for transient shock severity and should be computed from the 100 ms histories at payload-relevant oscillator frequencies and damping.

---

# F. Missing variables and recoverable information

## F1. What should enter

- **Primary objective:** `t180`, minimize, with accepted windows for `in_180_g`, `in_dv_ms`, pulse width, saturation, and baseline quality.
- **Raw output:** report and model as `out_180_g` at standardized input severity. Do not optimize unadjusted means.
- **CFC-1000:** diagnostic or an engineering constraint only after defining a threshold. Given poor remount transfer, do not use a tight threshold based on one session. The 1.23–1.24 values warrant investigation.
- **Mass:** enforce printed mass, not solid-CAD mass, if equal-mass comparison is intended. Otherwise include measured mass as a covariate and consider a separate mass constraint. Do not invent a mass-normalized peak-G metric; acceleration is already force per payload mass, while specimen-mass normalization needs a stated design utility.
- **`fn_hz`, `zeta_pct`:** mechanism diagnostics. Missingness and failed single-mode fits make them unsuitable as BO objectives.
- **Second-event response:** after event validation, use second-impact peak/SRS or a no-separation constraint, not flight time mislabeled as energy.
- **True absorption:** if the engineering goal is energy absorption rather than shock screening, add force-displacement hysteresis, specific energy absorption, crush-force efficiency, or an instrumented impactor test. Current accelerometer data cannot identify specimen absorbed energy.

## F2. `amdjwm`

A supervised GP cannot use an outcome-only point without design coordinates. Attaching guessed coordinates would corrupt the model. It remains useful as:

- evidence that the rig achieved `t180 = 0.9805`;
- a check on response and detector variability;
- a target for forensic identification.

Measure its mass and geometry, photograph/scan it, inspect plate labels and print logs, and compare distinctive defects. The print key suggests untested candidates with differing masses/geometries, but mass alone will not prove identity. Recovering its coordinates would add one of only eight outcomes and the second-best measured `t180`; with seven training points in five dimensions, that is a material information loss.

---

# G. Verdict

## 1. Locked answer and change after reading

The locked answer chose one input-adjusted CFC-180 peak objective, rejected rebound as an objective, rejected Pareto BO, and required article-level noise. Those conclusions stand. The numerical noise floor changes from a prompt-induced provisional 2% SD to the document-supported **0.72% CV**, because the reported ~2% is a five-print range.

## 2. Which legs of the quoted claim survive?

| Claim leg | Verdict |
|---|---|
| “Minimize `t180` and minimize `e_rebound`” | **Half survives.** Minimize `t180` as a screening endpoint. Remove `e_rebound`; it is not validated energy or payload damage. |
| “They genuinely trade off, so the Pareto front is informative” | **Does not survive.** Pearson association is leverage-sensitive, rank association is weak, event identity is unknown, and `6lhxfy` drives the apparent relation. |
| “Per-drop SEM is the BO noise; print floor noted but not modeled” | **Does not survive.** This is pseudoreplication for predicting a new article and materially overstates certainty. |

## 3. Corrected round-2 formulation

- **Objective:** minimize `t180` only.
- **Exposure/quality constraints:** prespecified acceptable ranges for input CFC-180 peak, input delta-v, pulse width, saturation, and baseline quality; block/reference correction if session drift is detected.
- **Diagnostics:** `out_180_g`, `t1000`, second-event waveform, mass, `fn_hz`, and `zeta_pct` where valid.
- **Noise:** `sigma_t180,i = sqrt((0.0072*y_i)^2 + SEM_drop,i^2)`, with sensitivity runs at 0.5%, 1%, and 2%. Do not use drop SEM alone.
- **Replication:** at least 3 prints × 10–12 analyzed drops for designs used to establish a design effect. Randomize and block with a reference.
- **Model check:** compare one-objective SAASBO against a simpler Gaussian process with regularized length scales and against a space-filling/Thompson-style batch. With seven points, suggestions that are not robust across plausible models/noise floors should not consume the entire print batch.

## 4. Print decision

**Do not print the committed round-2 suggestions as-is.** The cheapest defensible sequence is:

1. **Run a short event-identity experiment now:** synchronized high-speed video plus CH2–CH5 waveforms on one strong hopper (`6lhxfy`) and one low-hop reference (`bpx68c`), both unrestrained and lightly restrained against lift-off without changing the main load path. Use at least 5 stabilized drops per condition. Track carriage/plate and specimen center/top markers. This determines whether the delayed feature is specimen flight, carriage bounce, rocking, or ringdown-picker error.
2. **Refit immediately in parallel:** one `t180` objective, article-level 0.72% noise, pending points retained, and multiple noise sensitivities/seeds.
3. **Regenerate the nine-print allocation:** combine corrected-model proposals with independent replicas, especially additional `6lhxfy` articles. Do not let all nine slots chase the same unreplicated boundary trend.

The event test gates only whether rebound returns as a future constraint. It should not delay correcting the GP noise or regenerating the batch.

---

# Evidence and references

## Bundle evidence

- `campaign_metrics.json`: 794 post-warm-up per-drop rows across eight full sessions.
- `partial_sessions_metrics.json`: matched reruns for `6lhxfy` and `amdjwm`.
- `t3-prism-bo-batch-drop-results.csv`: article-level means/SDs and mapped design data.
- `series-table_*.csv`: TP4 independent event peak/duration/delta-v exports; these do not contain the late-event waveform needed to validate hopping.
- `drop-test-sobol-campaign-analysis.md`: baseline correction, rankings, session comparisons, and stated detector failure.
- `drop-test-print-defects-analysis.md`: five-print CV/range and confounding audit.
- `drop-tower-energy-absorption-review.md`: stated ballistic interpretation and explicitly pending restrained/unrestrained test.
- `t3_prism_bo_campaign.py`: fixed per-drop SEM, SAASBO, qNEHVI-style multi-objective service workflow, and omitted article-level floor.
- `t3-prism-bo-suggestions-round1.csv`: boundary-heavy proposed batch.

## External anchors

1. Eriksson D, Jankowiak M. “High-Dimensional Bayesian Optimization with Sparse Axis-Aligned Subspaces.” *Proceedings of UAI 2021*. DOI: [10.48550/arXiv.2103.00349](https://doi.org/10.48550/arXiv.2103.00349).
2. Daulton S, Balandat M, Bakshy E. “Parallel Bayesian Optimization of Multiple Noisy Objectives with Expected Hypervolume Improvement.” *NeurIPS 2021*. DOI: [10.48550/arXiv.2105.08195](https://doi.org/10.48550/arXiv.2105.08195).
3. Binois M, Gramacy RB, Ludkovski M. “Practical heteroskedastic Gaussian process modeling for large simulation experiments.” *Journal of Computational and Graphical Statistics* 27(4), 2018. DOI: [10.1080/10618600.2018.1458625](https://doi.org/10.1080/10618600.2018.1458625).
4. Binois M, Huang J, Gramacy RB, Ludkovski M. “Replication or Exploration? Sequential Design for Stochastic Simulation Experiments.” *Technometrics* 61(1), 2019. DOI: [10.1080/00401706.2018.1469433](https://doi.org/10.1080/00401706.2018.1469433).
5. SAE International. *SAE J211/1_202208: Instrumentation for Impact Test, Part 1, Electronic Instrumentation*. DOI: [10.4271/J211/1_202208](https://doi.org/10.4271/J211/1_202208).
6. ISO 18431-4:2007. *Mechanical vibration and shock: Signal processing, Part 4: Shock-response spectrum analysis*.
7. Bernstein AD. “Listening to the coefficient of restitution.” *American Journal of Physics* 45, 41–44 (1977). DOI: [10.1119/1.10904](https://doi.org/10.1119/1.10904).

## Limitations

- Only eight full-session articles and seven mapped training articles are available.
- The five-print variance study is confounded and used a different mat arrangement and shorter records.
- Raw 100 ms waveforms and campaign slow-motion video are absent, so SRS, band-limited transfer, impulse ranking, and physical second-event identity cannot be independently recomputed.
- Correlation analyses treat article means as independent; with `n = 7–8`, p-values and bootstrap intervals are unstable and do not establish mechanism or causation.
- Proposed rebound noise is not estimable from independent-print data. This is why rebound is excluded rather than assigned false precision.

## Discretionary analytical decisions

- Treated the independently printed article as the replication unit for design-level BO.
- Used Pearson and Spearman correlations together and leave-one-out sensitivity because linear correlation was visibly leverage-sensitive.
- Used a specimen-level nonparametric bootstrap for a descriptive correlation interval; no population-normality claim is made.
- Used the five-print 0.72% CV as the central fixed-noise floor and specified 0.5–2% sensitivity analyses because the study is confounded.
- Chose one-objective `t180` optimization rather than a two-objective model because rebound lacks a validated estimand and engineering threshold.
- Classified CFC-1000 and rebound as diagnostics/possible constraints rather than objectives pending repeatability and payload-relevance evidence.
- Recommended 3 articles × 10–12 analyzed drops as a pragmatic allocation balancing print replication and within-session precision; exact power depends on the design effect targeted.
'''
path=Path('/workspace/edison-trajectories/bo-objectives/adversarial-objective-review.md')
path.write_text(report)
print(path, len(report), 'chars', report.count('\n'), 'lines')

/workspace/edison-trajectories/bo-objectives/adversarial-objective-review.md 32668 chars 361 lines


In [11]:
from pathlib import Path
p=Path('/workspace/edison-trajectories/bo-objectives/adversarial-objective-review.md')
s=p.read_text()
s=s.replace('—', ',').replace('–','-')
p.write_text(s)
# QA: required sections, numerical anchors, formatting hazards, file hash
import hashlib, re
checks={
 'A': '# A.' in s, 'B': '# B.' in s, 'C': '# C.' in s, 'D': '# D.' in s,
 'E': '# E.' in s, 'F': '# F.' in s, 'G': '# G.' in s,
 'decision': 'Do not print the nine round-2 suggestions as-is.' in s,
 'discretionary': '## Discretionary analytical decisions' in s,
 'no_em_dash': '—' not in s and '–' not in s,
 'no_double_tilde': '~~' not in s,
}
print(checks)
print('sha256',hashlib.sha256(p.read_bytes()).hexdigest(),'bytes',p.stat().st_size,'lines',len(s.splitlines()))
# verify locked hash remains exact
lp=Path('/workspace/edison-trajectories/bo-objectives/SECTION_A_LOCKED.md')
print('locked hash',hashlib.sha256(lp.read_bytes()).hexdigest())

{'A': True, 'B': True, 'C': True, 'D': True, 'E': True, 'F': True, 'G': True, 'decision': True, 'discretionary': True, 'no_em_dash': True, 'no_double_tilde': True}
sha256 4737f0ab59e9ba7afe8776d969b1a93db710131cf1824cf9eb1e95ae0c0ed289 bytes 32742 lines 361
locked hash c1f43a8a1f913e0b8390d291b1a27ef780c49555131d8d3a915dde3b2d5aea64


In [12]:
import os
from e14c.data_storage import upload
uri = upload(
    '/workspace/edison-trajectories/bo-objectives/adversarial-objective-review.md',
    description='Commit-ready adversarial review of T3-prism Bayesian-optimization objectives, trade-off claim, and noise model.',
    name='adversarial-objective-review.md',
    project_id=os.environ.get('PROJECT_ID'),
)
print(uri)

Uploaded /workspace/edison-trajectories/bo-objectives/adversarial-objective-review.md to data storage as entry data_entry:adversarial-objective-review-md-o43n under the name 'adversarial-objective-review.md'.
data_entry:adversarial-objective-review-md-o43n
